# YouTube keyword and tag research: autocomplete + competitor tags

Companion notebook for the MLforSEO guides **[YouTube SEO: how to boost video visibility and grow your channel](https://www.mlforseo.com/machine-learning-implementation-guides/content-creation/youtube-seo-guide/)** and **[How to do a YouTube SEO audit](https://www.mlforseo.com/machine-learning-implementation-guides/content-audit/youtube-seo-audit/)**, by Lazarina Stoy.

"YouTube tag generator" sites give you a list of words and no reasons. This notebook does the research behind a tag list, with data you can check:

1. **YouTube autocomplete** for your seed terms, expanded with a-z, question and modifier prefixes, then grouped into topics.
2. **The top videos for a seed** from the official **YouTube Data API v3**: titles, tags, views, publish date, duration.
3. **Tag frequency and co-occurrence**: which tags the top videos share, and which travel together.
4. **Title n-grams and patterns**: the words, phrases and formats that top titles use.
5. **Gaps**: autocomplete phrases that none of the top videos target in their title or tags.
6. **A suggested tag list** that fits YouTube's 500-character limit, with the reason for each tag.
7. **Export** to CSV and a Sheets-ready Excel file.

**A word on tags.** YouTube's own help centre says your title, thumbnail and description "are more important pieces of metadata for your video's discovery", and that tags "play a minimal role" unless your topic is commonly misspelled ([YouTube Help: add tags to your video](https://support.google.com/youtube/answer/146402)). So use this mostly for **keyword and title research**; the tag list is the small final step.

## What you need

* **Sample mode (default):** nothing. It uses real autocomplete responses saved on 24 October 2026 and a **sample** set of YouTube Data API responses (fictional channels and fake video IDs, in the official response format).
* **Live mode:** internet access for autocomplete, and a free **YouTube Data API v3 key** for the competitor part: Google Cloud console > APIs & Services > Library > enable *YouTube Data API v3* > Credentials > Create credentials > API key. Restrict the key to that API. In Colab, save it under the key icon (Secrets) as `YOUTUBE_API_KEY`.

**Cost:** the API is free within the default quota. Per seed this notebook makes **1 search.list call** (search has its own bucket of 100 calls a day, 1 unit per call) and **1 videos.list call** (1 unit from the general 10,000 units a day; one call covers up to 50 videos). Source: [YouTube Data API quota costs](https://developers.google.com/youtube/v3/determine_quota_cost), updated 15 October 2026. Quotas reset at midnight Pacific Time.

**The autocomplete endpoint is undocumented.** `suggestqueries.google.com` is what YouTube's search box calls, but Google doesn't document or support it for developers. It can change, rate-limit or block you without notice. The code pauses between calls, records every failure, and stops sending after 5 failures in a row so the rest of the notebook still runs.

In [1]:
import os, sys, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Download the helper and the sample files if they aren't next to the notebook (e.g. in Colab)
RAW = "https://raw.githubusercontent.com/lazarinastoy/mlforseo-code/main/youtube-keyword-research/"
for f in ["youtube_keyword_research.py", "sample_data/autocomplete_sample.json",
          "sample_data/search_list_youtube-seo.json", "sample_data/videos_list_youtube-seo.json"]:
    if not Path(f).exists():
        Path(f).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(RAW + f, f)

import pandas as pd
import youtube_keyword_research as ykr
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("Ready.")

Ready.


## 1. Settings

* `SEEDS`: the topics you want to research. Two to five is plenty; each seed sends about 41 autocomplete requests.
* `COMPETITOR_SEEDS`: the seeds you want to pull top videos for (each costs 2 API calls).
* `HL` and `GL`: interface language and country for autocomplete. `REGION` and `LANGUAGE` do the same for the API search.

In [2]:
USE_SAMPLE = True

SEEDS = ["youtube seo", "youtube tags"]
COMPETITOR_SEEDS = ["youtube seo"]
HL, GL = "en", "GB"              # autocomplete language and country
REGION, LANGUAGE = "GB", "en"    # API search region and relevance language
TOP_N_VIDEOS = 20                # 1-50 videos per seed, still one API call either way

SAMPLE_TODAY = pd.Timestamp("2026-10-24", tz="UTC")  # keeps sample-mode numbers stable

## 2. YouTube autocomplete

For each seed the notebook asks autocomplete for the seed itself, the seed followed by each letter a to z, question prefixes (*how to*, *what is*, *why*, *best*...) and common modifiers (*for beginners*, *tutorial*, *tools*, *2026*...). You can edit `QUESTION_PREFIXES` and `SUFFIX_MODIFIERS` in the helper.

`times_suggested` counts how many different expansions returned a phrase. It is a rough signal of how strongly YouTube associates the phrase with the seed, not a search volume.

In [3]:
if USE_SAMPLE:
    cache = ykr.load_json("sample_data/autocomplete_sample.json")["responses"]
    print(ykr.load_json("sample_data/autocomplete_sample.json")["_note"])
else:
    cache = None

auto_raw, auto_cache, auto_errors = ykr.expand_seeds(SEEDS, hl=HL, gl=GL, cache=cache)
suggestions = ykr.group_suggestions(ykr.unique_suggestions(auto_raw))

print(f"{len(suggestions)} unique suggestions from {auto_raw['query_sent'].nunique()} autocomplete queries")
if auto_errors:
    print(f"{len(auto_errors)} autocomplete requests failed; first few:")
    display(pd.DataFrame(auto_errors).head())
suggestions.head(20)

SAMPLE: real YouTube autocomplete responses collected on 24 October 2026 (hl=en, gl=GB) from the undocumented suggestqueries endpoint. Suggestions change daily.
539 unique suggestions from 82 autocomplete queries


,seed,suggestion,times_suggested,best_rank,modifier,topic_group
136,youtube seo,youtube seo crash course,8,2,crash course,learning / course
135,youtube seo,youtube seo course in urdu,4,3,course in urdu,language / region
93,youtube seo,youtube seo ai tool,3,1,ai tool,tools / software
97,youtube seo,youtube seo and monetization,3,1,and monetization,career / money
129,youtube seo,youtube seo course,3,1,course,learning / course
147,youtube seo,youtube seo for beginners,3,1,for beginners,learning / course
151,youtube seo,youtube seo full course,3,1,full course,learning / course
197,youtube seo,youtube seo live project,3,1,live project,other
213,youtube seo,youtube seo optimization,3,1,optimization,tips / tactics
243,youtube seo,youtube seo tips,3,1,tips,tips / tactics


### Topic groups

Suggestions are grouped by what they add to the seed (the *modifier*) using simple, readable rules: questions, learning and courses, tools, tips, year, language or region, career and money, comparisons, formats and features. Edit `topic_bucket()` in the helper to fit your niche. Rules beat a black-box clustering model here because you can see exactly why a phrase landed in a group.

In [4]:
summary = (suggestions.groupby(["seed", "topic_group"])
           .agg(phrases=("suggestion", "count"),
                examples=("suggestion", lambda s: " | ".join(s.head(4))))
           .reset_index()
           .sort_values(["seed", "phrases"], ascending=[True, False]))
summary

,seed,topic_group,phrases,examples
5,youtube seo,other,103,youtube seo live project | youtube video seo | youtube seo manoj dey | youtube seo by ...
3,youtube seo,language / region,59,youtube seo course in urdu | youtube seo full course in telugu | youtube seo bangla | ...
6,youtube seo,question,34,how youtube seo works | what is youtube seo | how to do youtube seo with chatgpt | how...
4,youtube seo,learning / course,25,youtube seo crash course | youtube seo course | youtube seo for beginners | youtube se...
9,youtube seo,tools / software,25,youtube seo ai tool | youtube seo tools | youtube seo with ai | best youtube seo tools
10,youtube seo,year / freshness,20,youtube seo 2026 | youtube seo tutorial 2025 | youtube seo 2026 hamouda courses | yout...
2,youtube seo,format / feature,13,youtube seo keywords in description | youtube seo keyword research | youtube seo descr...
8,youtube seo,tips / tactics,12,youtube seo optimization | youtube seo tips | youtube seo search engine optimization |...
0,youtube seo,career / money,10,youtube seo and monetization | youtube seo client hunting | youtube seo jobs in hydera...
1,youtube seo,comparison,6,youtube seo vs wordpress | best youtube seo | youtube seo vs youtube | youtube seo vs ...


## 3. Competitor videos (YouTube Data API v3)

`search.list` returns the top videos for each competitor seed (ordered by relevance, the closest the API gets to YouTube's search ranking; results can differ from what you see logged in). `videos.list` then adds title, tags, view, like and comment counts, publish date and duration.

The API key is read at runtime: Colab Secrets first, then the `YOUTUBE_API_KEY` environment variable, then a hidden prompt. It is never written to the notebook.

In [5]:
def get_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("YOUTUBE_API_KEY")
        if key:
            return key
    except Exception:
        pass
    if os.environ.get("YOUTUBE_API_KEY"):
        return os.environ["YOUTUBE_API_KEY"]
    from getpass import getpass
    return getpass("YouTube Data API v3 key: ")

frames = []
if USE_SAMPLE:
    print(ykr.load_json("sample_data/videos_list_youtube-seo.json")["_sample_note"])
    for seed in COMPETITOR_SEEDS:
        s_json = ykr.load_json(f"sample_data/search_list_{ykr.slug(seed)}.json")
        v_json = ykr.load_json(f"sample_data/videos_list_{ykr.slug(seed)}.json")
        frames.append(ykr.videos_frame(seed, s_json, v_json, today=SAMPLE_TODAY))
else:
    API_KEY = get_api_key()
    for seed in COMPETITOR_SEEDS:
        try:
            s_json = ykr.search_videos(seed, API_KEY, max_results=TOP_N_VIDEOS, region=REGION, language=LANGUAGE)
            ids = [it["id"]["videoId"] for it in s_json.get("items", []) if it["id"].get("videoId")]
            v_json = ykr.video_details(ids, API_KEY)
            # keep the raw responses so you can re-run the analysis without spending quota
            ykr.save_json(s_json, f"my_data/search_list_{ykr.slug(seed)}.json")
            ykr.save_json(v_json, f"my_data/videos_list_{ykr.slug(seed)}.json")
            frames.append(ykr.videos_frame(seed, s_json, v_json))
        except ykr.YouTubeAPIError as e:
            print("API error for", repr(seed), "->", e)

videos = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
videos[["seed", "search_rank", "title", "channel", "published", "views", "views_per_day", "n_tags", "tags_text"]]

SAMPLE DATA, not real YouTube results: fictional channels and fake video IDs, written by MLforSEO to match the official YouTube Data API v3 response shape. Run the notebook with your own API key for real data.


,seed,search_rank,title,channel,published,views,views_per_day,n_tags,tags_text
0,youtube seo,1,YouTube SEO 2026: How to Rank Videos in Search (Step by Step),Creator Lab Academy,2026-01-14,412000,1461.0,8,"youtube seo, youtube seo 2026, how to rank youtube videos, youtube seo tips, youtube a..."
1,youtube seo,2,YouTube SEO for Beginners: The Complete Guide,Small Channel Studio,2025-09-03,286000,689.2,7,"youtube seo, youtube seo for beginners, youtube seo tutorial, youtube seo full course,..."
2,youtube seo,3,7 YouTube SEO Tips That Actually Work,Video Marketing Weekly,2025-11-20,198000,587.5,6,"youtube seo, youtube seo tips, youtube tips, youtube growth, get more views on youtube..."
3,youtube seo,4,How to Find Keywords for YouTube Videos (Free Tools),Creator Lab Academy,2025-06-11,156000,312.6,6,"youtube keyword research, youtube seo, youtube keyword tool, free youtube seo tools, y..."
4,youtube seo,5,YouTube Tags: Do They Still Matter? (I Tested It),The Upload Test,2025-12-02,121000,372.3,6,"youtube tags, youtube seo, do youtube tags matter, youtube tags 2026, youtube metadata..."
5,youtube seo,6,"YouTube SEO Crash Course | Titles, Descriptions, Tags & Chapters",Digital Skills Hub,2026-03-09,98000,429.8,7,"youtube seo, youtube seo crash course, youtube seo full course, youtube title, youtube..."
6,youtube seo,7,Best YouTube SEO Tools in 2026 (Free and Paid),Tool Review Desk,2026-02-18,87000,352.2,6,"best youtube seo tools, youtube seo tools, youtube seo, youtube seo 2026, youtube keyw..."
7,youtube seo,8,How YouTube Search Works (Explained by a Former Ads Analyst),Signal & Noise Media,2025-08-27,76000,180.1,5,"how youtube seo works, youtube search, youtube algorithm, youtube seo, youtube recomme..."
8,youtube seo,9,YouTube SEO with AI: Write Titles and Descriptions in Minutes,AI Workflow Studio,2026-05-06,64000,376.5,6,"youtube seo with ai, youtube seo ai tool, chatgpt for youtube, youtube seo, youtube ti..."
9,youtube seo,10,YouTube SEO Checklist: 20 Things to Do Before You Hit Publish,Video Marketing Weekly,2026-04-01,52000,253.7,6,"youtube seo checklist, youtube seo, youtube upload checklist, youtube seo tips, youtub..."


## 4. Tags: frequency and co-occurrence

* `videos_using` and `share_of_videos`: how many of the top videos use the tag.
* `total_views` / `median_views`: the views behind those videos. A tag used by one huge video can look important when it isn't; prefer tags used by several.
* Co-occurrence shows tags that travel together, which is a decent map of how creators in the niche describe the topic.

Remember that only the uploader sees tags in YouTube Studio; the API returns them for public videos, and some creators don't use tags at all (`n_tags = 0`).

In [6]:
tags = ykr.tag_stats(videos)
tags.head(20)

,tag,videos_using,total_views,median_views,best_rank,share_of_videos
0,youtube seo,19,2458000,98000.0,1,0.95
1,youtube algorithm,5,1093000,198000.0,1,0.25
2,youtube seo tips,5,891000,168000.0,1,0.25
3,youtube growth,3,753000,198000.0,1,0.15
4,youtube keyword research,3,626000,156000.0,1,0.15
5,youtube seo 2026,3,557000,87000.0,1,0.15
6,youtube seo full course,3,517000,133000.0,2,0.15
7,youtube search,3,290000,76000.0,4,0.15
8,youtube description,3,223000,64000.0,6,0.15
9,rank youtube videos,2,580000,290000.0,1,0.10


In [7]:
cooc = ykr.tag_cooccurrence(videos, min_count=2)
cooc.head(15)

,tag_a,tag_b,videos_together
0,youtube algorithm,youtube seo,5
1,youtube seo,youtube seo tips,5
2,youtube search,youtube seo,3
3,youtube seo,youtube seo full course,3
4,youtube description,youtube seo,3
5,youtube growth,youtube seo,3
6,youtube keyword research,youtube seo,3
7,youtube seo,youtube seo 2026,3
8,keyword research,youtube search,2
9,keyword research,youtube seo,2


## 5. Titles: n-grams and patterns

Titles matter far more than tags for discovery, so this is the most useful part of the competitor data. N-grams are the words (n=1) and phrases (n=2, 3) that appear in the most top titles. The pattern table compares median views for titles with and without each pattern; with 20 videos per seed, treat it as a description of the niche, not proof of what works.

In [8]:
ngrams = ykr.title_ngrams(videos, top=15)
ngrams[ngrams["n"] > 1].sort_values(["titles", "total_views"], ascending=False).head(20)

,n,ngram,titles,total_views
15,2,youtube seo,11,1424000
30,3,step by step,2,470000
16,2,youtube search,2,244000
17,2,rank videos,1,412000
18,2,seo 2026,1,412000
19,2,search step,1,412000
31,3,videos in search,1,412000
32,3,youtube seo 2026,1,412000
20,2,complete guide,1,286000
33,3,seo for beginners,1,286000


In [9]:
patterns = ykr.title_patterns(videos)
patterns

,pattern,share_of_titles,median_views_with,median_views_without
0,contains a number,0.45,133000,76000.0
1,contains a year (20xx),0.15,87000,98000.0
2,starts with How / What / Why,0.15,76000,98000.0
3,uses brackets ( ) or [ ],0.40,104000,81000.0
4,uses a colon or pipe,0.55,98000,87000.0
5,says 'for beginners',0.10,209500,81500.0
6,all-caps word (3+ letters),0.80,92500,98500.0
7,median title length (characters),NaN,50,NaN


## 6. Gaps: what people search that top videos don't target

An autocomplete phrase is a **gap** when none of the top videos for that seed uses it in the title or tags. The most useful gaps are the ones suggested by several expansions (`times_suggested` of 2 or more). Language and region variants show up a lot for English seeds; filter them out if they're not your audience.

In [10]:
gaps = pd.concat([ykr.coverage_gaps(suggestions[suggestions["seed"] == s], videos[videos["seed"] == s])
                  for s in COMPETITOR_SEEDS], ignore_index=True)
top_gaps = (gaps[gaps["gap"] & (gaps["times_suggested"] >= 2) & (gaps["topic_group"] != "language / region")]
            .sort_values("times_suggested", ascending=False))
top_gaps[["seed", "suggestion", "times_suggested", "topic_group"]].head(20)

,seed,suggestion,times_suggested,topic_group
3,youtube seo,youtube seo and monetization,3,career / money
8,youtube seo,youtube seo optimization,3,tips / tactics
11,youtube seo,youtube seo client hunting,3,career / money
13,youtube seo,youtube seo keywords in description,3,format / feature
15,youtube seo,youtube seo search engine optimization,3,tips / tactics
7,youtube seo,youtube seo live project,3,other
66,youtube seo,how to do youtube seo 2026,2,question
58,youtube seo,how to do youtube seo with chatgpt,2,question
59,youtube seo,how to seo youtube video,2,question
60,youtube seo,seo youtube là gì,2,other


## 7. A suggested tag list

The list starts with the seed, then phrases that autocomplete suggests and at least two top videos use as tags, then other tags shared by at least two top videos, then the strongest remaining autocomplete phrases. It stops at 15 tags or YouTube's 500-character limit, counted the way YouTube counts it: commas count, and a tag with a space counts as if wrapped in quotes ([videos resource, `snippet.tags`](https://developers.google.com/youtube/v3/docs/videos)).

Edit it by hand: drop anything your video doesn't actually cover, and add a common misspelling if your topic has one (the one case where YouTube says tags help).

In [11]:
tag_lists = {}
for seed in COMPETITOR_SEEDS:
    t = ykr.tag_stats(videos[videos["seed"] == seed])
    tag_lists[seed] = ykr.suggest_tags(seed, suggestions, t)
    chars = ykr.tag_field_length(tag_lists[seed]["tag"].tolist())
    print(f"{seed}: {len(tag_lists[seed])} tags, {chars}/500 characters")
    print(", ".join(tag_lists[seed]["tag"]))
suggested_tags = pd.concat([df.assign(seed=s) for s, df in tag_lists.items()], ignore_index=True)
suggested_tags

youtube seo: 15 tags, 313/500 characters
youtube seo, youtube seo tips, youtube seo full course, youtube seo 2026, youtube seo ai tool, youtube seo for beginners, youtube seo tutorial, youtube algorithm, youtube growth, youtube keyword research, youtube search, youtube description, rank youtube videos, youtube keyword tool, youtube tips


,tag,why,used_by_top_videos,autocomplete_hits,seed
0,youtube seo,seed,19,1,youtube seo
1,youtube seo tips,autocomplete + competitor tag,5,3,youtube seo
2,youtube seo full course,autocomplete + competitor tag,3,3,youtube seo
3,youtube seo 2026,autocomplete + competitor tag,3,2,youtube seo
4,youtube seo ai tool,autocomplete + competitor tag,2,3,youtube seo
5,youtube seo for beginners,autocomplete + competitor tag,2,3,youtube seo
6,youtube seo tutorial,autocomplete + competitor tag,2,2,youtube seo
7,youtube algorithm,competitor tag (2+ top videos),5,0,youtube seo
8,youtube growth,competitor tag (2+ top videos),3,0,youtube seo
9,youtube keyword research,competitor tag (2+ top videos),3,0,youtube seo


## 8. Export

One CSV per table plus `youtube_keyword_research.xlsx` with a tab for each. Upload the `.xlsx` to Google Drive and open it with Google Sheets.

In [12]:
written = ykr.export({
    "suggestions": suggestions,
    "topic_groups": summary,
    "videos": videos,
    "tags": tags,
    "tag_cooccurrence": cooc,
    "title_ngrams": ngrams,
    "title_patterns": patterns,
    "gaps": gaps,
    "suggested_tags": suggested_tags,
}, "results")
for p in written:
    print(p)

if IN_COLAB:
    from google.colab import files
    files.download("results/youtube_keyword_research.xlsx")

results/youtube_suggestions.csv
results/youtube_topic_groups.csv
results/youtube_videos.csv
results/youtube_tags.csv
results/youtube_tag_cooccurrence.csv
results/youtube_title_ngrams.csv
results/youtube_title_patterns.csv
results/youtube_gaps.csv
results/youtube_suggested_tags.csv
results/youtube_keyword_research.xlsx


## Limits and honest caveats

* **Autocomplete is not search volume.** It shows what YouTube predicts people type, shaped by language, location and freshness. Suggestions change daily.
* **The suggest endpoint is undocumented** and may break or block you. If it fails, wait and re-run, or lower the number of seeds.
* **API search results are not the logged-in results** you see on YouTube, which are personalised. Treat them as a fair sample of what ranks.
* **Tags are a minor signal.** YouTube says so itself. Spend your time on the title, thumbnail, description, chapters and the video.
* **Small samples.** 20 to 50 videos per seed describe a niche; they don't prove what causes views.
* **Sample mode uses fictional API data.** The autocomplete sample is real (24 October 2026); the video data is not.

## What to do next

* Turn the gaps and title phrases into a content plan, then run each new video through the free **[YouTube SEO Checklist](https://www.mlforseo.com/resource/youtube-seo-checklist/)** (72 steps).
* Read **[YouTube SEO: how to boost video visibility and grow your channel](https://www.mlforseo.com/machine-learning-implementation-guides/content-creation/youtube-seo-guide/)** for the full workflow, and **[How to do a YouTube SEO audit](https://www.mlforseo.com/machine-learning-implementation-guides/content-audit/youtube-seo-audit/)** to benchmark your channel against the competitors you found here.
* For Google Search and Places autocomplete as well as YouTube, see the **[multi-platform autocomplete notebook](https://www.mlforseo.com/resource/multi-platform-autocomplete-keyword-research-with-google-search-youtube-and-places-apis-notebook/)**.
* Build this into a full keyword universe in the course **[Semantic AI-powered SEO Keyword Research](https://www.mlforseo.com/courses/semantic-ml-enabled-keyword-research/)**.